# Experiment 7: AutoGluon feature-subset comparison

This experiment compares the full shared-preprocessor feature set against correlation-guided subsets. Every model uses the same chronological validation split and the same untuned GBM plus CatBoost AutoGluon ensemble. Each subset is then refit on all labelled rows, and the final cell compares their modified-test proxy RMSE.

In [1]:
# If needed: pip install autogluon.tabular
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

configured_root = os.environ.get('KDDM_PROJECT_ROOT')
candidate_roots = ([Path(configured_root)] if configured_root else []) + [
    Path.cwd(), *Path.cwd().parents, Path('/content/KDDM-project'),
    Path('/content/drive/MyDrive/KDDM-project'),
]
project_root = next((p for p in candidate_roots if (p / 'exploration/metul/preprocessing.py').exists()), None)
if project_root is None:
    raise FileNotFoundError('Set KDDM_PROJECT_ROOT to the project directory.')
experiment_dir = project_root / 'exploration' / 'metul'
sys.path.insert(0, str(experiment_dir))

from autogluon.tabular import TabularPredictor
from preprocessing import RentalPreprocessor, load_datasets

RANDOM_STATE = 42
TARGET = 'MONTHLY_RENT'
TIME_LIMIT_SECONDS = int(os.environ.get('AUTOGLUON_TIME_LIMIT', '600'))
RUN_ID = os.environ.get('AUTOGLUON_RUN_ID', pd.Timestamp.now().strftime('%Y%m%d_%H%M%S'))

# Deliberately no hyperparameter tuning: use AutoGluon's default GBM/CatBoost settings.
MODEL_HYPERPARAMETERS = {'GBM': {}, 'CAT': {}}


In [2]:
data_dir, datasets = load_datasets(project_root / 'data')
raw_train = datasets['train'].copy()
raw_test = datasets['test'].copy()
raw_train['period'] = pd.to_datetime(raw_train['RENT_APPROVAL_DATE'] + '-01')
raw_test['period'] = pd.to_datetime(raw_test['RENT_APPROVAL_DATE'] + '-01')

# Same 80/20 chronological validation convention as the earlier AutoGluon experiments.
periods = np.sort(raw_train['period'].unique())
validation_cutoff = pd.Timestamp(periods[int(len(periods) * 0.80)])
raw_fit = raw_train.loc[raw_train['period'] < validation_cutoff].drop(columns='period').copy()
raw_valid = raw_train.loc[raw_train['period'] >= validation_cutoff].drop(columns='period').copy()

# Fit preprocessing only on the earlier validation period, then fit separately on all train rows for test predictions.
validation_preprocessor = RentalPreprocessor(datasets)
fit_df = validation_preprocessor.fit_transform(raw_fit)
valid_df = validation_preprocessor.transform(raw_valid)

final_preprocessor = RentalPreprocessor(datasets)
full_train_df = final_preprocessor.fit_transform(raw_train.drop(columns='period'))
test_df = final_preprocessor.transform(raw_test.drop(columns='period'))

print(f'Validation cutoff: {validation_cutoff:%Y-%m}')
print(f'Fit rows: {fit_df.shape}; validation rows: {valid_df.shape}')
print(f'Full train: {full_train_df.shape}; test: {test_df.shape}')


[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces


In [3]:
def prepare_subset(frame, feature_columns):
    result = frame.loc[:, feature_columns].copy()
    if 'RENT_APPROVAL_DATE' in result.columns:
        result['RENT_APPROVAL_DATE'] = pd.to_datetime(result['RENT_APPROVAL_DATE'] + '-01')
    return result

all_preprocessed_features = [column for column in full_train_df.columns if column != TARGET]

# Retains the strongest, non-duplicative signals from the correlation audit.
compact_nonredundant_features = [
    'TOWN', 'BLOCK', 'STREET', 'FLAT_TYPE', 'FLAT_MODEL',
    'FLOOR_AREA_SQM', 'approval_date_num', 'approval_month',
    'coe_quota_premium_mean', 'YEAR_COMPLETED', 'MAX_FLOOR',
    'LATITUDE', 'LONGITUDE', 'mrt_dist_nearest_km', 'mall_dist_nearest_km',
]

# Future-only ablations: approval_month is retained for recurring seasonality.
compact_no_approval_date_features = [
    column for column in compact_nonredundant_features
    if column != 'approval_date_num'
]
compact_no_coordinates_features = [
    column for column in compact_nonredundant_features
    if column not in {'LATITUDE', 'LONGITUDE'}
]

# Adds weaker but plausible candidates as one group, so their net validation effect is measurable.
compact_plus_candidates_features = compact_nonredundant_features + [
    'REGION', 'stock_close_mean', 'stock_volume_mean',
    'mrt_within_1km', 'mrt_within_2km',
    'mall_within_1km', 'mall_within_2km',
    'school_dist_nearest_km', 'school_within_1km', 'school_within_2km',
]

FEATURE_SUBSETS = {
    'full_preprocessed': all_preprocessed_features,
    'compact_nonredundant': compact_nonredundant_features,
    # 'compact_no_approval_date': compact_no_approval_date_features,
    'compact_no_coordinates': compact_no_coordinates_features,
    'compact_plus_candidates': compact_plus_candidates_features,
}

for subset_name, columns in FEATURE_SUBSETS.items():
    missing = set(columns) - set(full_train_df.columns) - {TARGET}
    if missing:
        raise ValueError(f'{subset_name} references missing preprocessed features: {sorted(missing)}')
    if len(columns) != len(set(columns)):
        raise ValueError(f'{subset_name} contains duplicate feature names.')

subset_summary = pd.DataFrame([
    {'feature_subset': name, 'feature_count': len(columns), 'features': ', '.join(columns)}
    for name, columns in FEATURE_SUBSETS.items()
])
display(subset_summary)

y_fit = fit_df[TARGET].astype(float)
y_valid = valid_df[TARGET].astype(float)
y_full = full_train_df[TARGET].astype(float)


,feature_subset,feature_count,features
0,full_preprocessed,31,"RENT_APPROVAL_DATE, TOWN, BLOCK, STREET, FLAT_..."
1,compact_nonredundant,15,"TOWN, BLOCK, STREET, FLAT_TYPE, FLAT_MODEL, FL..."
2,compact_no_coordinates,13,"TOWN, BLOCK, STREET, FLAT_TYPE, FLAT_MODEL, FL..."
3,compact_plus_candidates,25,"TOWN, BLOCK, STREET, FLAT_TYPE, FLAT_MODEL, FL..."


In [4]:
validation_results = []
validation_predictors = {}

for subset_name, columns in FEATURE_SUBSETS.items():
    X_fit = prepare_subset(fit_df, columns)
    X_valid = prepare_subset(valid_df, columns)
    predictor = TabularPredictor(
        label=TARGET,
        problem_type='regression',
        eval_metric='root_mean_squared_error',
        path=str(experiment_dir / 'ag_feature_subset_models' / RUN_ID / f'validation_{subset_name}'),
    ).fit(
        train_data=X_fit.assign(**{TARGET: y_fit.to_numpy()}),
        tuning_data=X_valid.assign(**{TARGET: y_valid.to_numpy()}),
        hyperparameters=MODEL_HYPERPARAMETERS,
        presets='medium_quality',
        time_limit=TIME_LIMIT_SECONDS,
        verbosity=2,
    )
    predictions = predictor.predict(X_valid).astype(float).clip(lower=0)
    validation_predictors[subset_name] = predictor
    validation_results.append({
        'feature_subset': subset_name,
        'feature_count': len(columns),
        'RMSE': mean_squared_error(y_valid, predictions) ** 0.5,
        'MAE': mean_absolute_error(y_valid, predictions),
        'R2': r2_score(y_valid, predictions),
    })

validation_metrics = pd.DataFrame(validation_results).sort_values('RMSE').reset_index(drop=True)
display(validation_metrics)
print('Select the preferred subset from chronological validation before interpreting modified-test proxy RMSE.')


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       4.03 GB / 16.00 GB (25.2%)
Disk Space Avail:   191.18 GB / 460.38 GB (41.5%)
Presets specified: ['medium_quality']
Beginning AutoGluon training ... Time limit = 600s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_feature_subset_models/20260922_171439/validation_full_preprocessed"
Train Data Rows:    117988
Train Data Columns: 31
Tuning Data Rows:    32012
Tuning Data Columns: 31
Label Column:       MONTHLY_RENT
Problem Type:       regre

,feature_subset,feature_count,RMSE,MAE,R2
0,compact_no_coordinates,13,467.230403,345.659305,0.494243
1,compact_nonredundant,15,467.480018,345.786155,0.493703
2,full_preprocessed,31,467.507114,346.646881,0.493644
3,compact_plus_candidates,25,468.303867,348.313518,0.491917


Select the preferred subset from chronological validation before interpreting modified-test proxy RMSE.


In [5]:
test_prediction_frame = pd.DataFrame({'Id': raw_test.index})
competition_submissions = {}

for subset_name, columns in FEATURE_SUBSETS.items():
    X_full = prepare_subset(full_train_df, columns)
    X_test = prepare_subset(test_df, columns)
    predictor = TabularPredictor(
        label=TARGET,
        problem_type='regression',
        eval_metric='root_mean_squared_error',
        path=str(experiment_dir / 'ag_feature_subset_models' / RUN_ID / f'final_{subset_name}'),
    ).fit(
        train_data=X_full.assign(**{TARGET: y_full.to_numpy()}),
        hyperparameters=MODEL_HYPERPARAMETERS,
        presets='medium_quality',
        time_limit=TIME_LIMIT_SECONDS,
        verbosity=2,
    )
    predictions = predictor.predict(X_test).astype(float).clip(lower=0)
    test_prediction_frame[subset_name] = predictions.to_numpy()
    submission = pd.DataFrame({'Id': raw_test.index, 'Predicted': predictions.to_numpy()})
    competition_submissions[subset_name] = submission
    submission_path = experiment_dir / f'exp_7_autogluon_{subset_name}_submission.csv'
    submission.to_csv(submission_path, index=False)
    print(f'Saved {subset_name} submission to {submission_path}')

comparison_prediction_path = experiment_dir / 'exp_7_autogluon_feature_subset_test_predictions.csv'
test_prediction_frame.to_csv(comparison_prediction_path, index=False)
display(test_prediction_frame.head())


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       3.72 GB / 16.00 GB (23.3%)
Disk Space Avail:   190.95 GB / 460.38 GB (41.5%)
Presets specified: ['medium_quality']
Beginning AutoGluon training ... Time limit = 600s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_feature_subset_models/20260922_171439/final_full_preprocessed"
Train Data Rows:    150000
Train Data Columns: 31
Label Column:       MONTHLY_RENT
Problem Type:       regression
Preprocessing data...
Using Feature Generators to 

Saved full_preprocessed submission to /Users/metul/Desktop/KDDM-project/exploration/metul/exp_7_autogluon_full_preprocessed_submission.csv


	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
		Fitting CategoryFeatureGenerator...
			Fitting CategoryMemoryMinimizeFeatureGenerator...
		Fitting TextSpecialFeatureGenerator...
			Fitting BinnedFeatureGenerator...
			Fitting DropDuplicatesFeatureGenerator...
		Fitting TextNgramFeatureGenerator...
			Fitting CountVectorizer for text features: ['STREET']
			CountVectorizer fit with vocabulary size = 9
		Reducing Vectorizer vocab size from 9 to 3 to avoid OOM error
	Stage 4 Generators:
		Fitting DropUniqueFeatureGenerator...
	Stage 5 Generators:
		Fitting DropDuplicatesFeatureGenerator...
	Types of features in original data (raw dtype, special dtypes):
		('float', [])        : 6 | ['FLOOR_AREA_SQM', 'coe_quota_premium_mean', 'LATITUDE', 'LONGITUDE', 'mrt_dist_nearest_km', ...]
		('int', [])          : 4 | ['approval_date_num', 'approval_month', 'YEAR_COMPLETED', 'MAX_FLOOR']
		('object', [])       : 4 | ['TOWN', 'BLOCK

Saved compact_nonredundant submission to /Users/metul/Desktop/KDDM-project/exploration/metul/exp_7_autogluon_compact_nonredundant_submission.csv


	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
		Fitting CategoryFeatureGenerator...
			Fitting CategoryMemoryMinimizeFeatureGenerator...
		Fitting TextSpecialFeatureGenerator...
			Fitting BinnedFeatureGenerator...
			Fitting DropDuplicatesFeatureGenerator...
		Fitting TextNgramFeatureGenerator...
			Fitting CountVectorizer for text features: ['STREET']
			CountVectorizer fit with vocabulary size = 9
		Reducing Vectorizer vocab size from 9 to 3 to avoid OOM error
	Stage 4 Generators:
		Fitting DropUniqueFeatureGenerator...
	Stage 5 Generators:
		Fitting DropDuplicatesFeatureGenerator...
	Types of features in original data (raw dtype, special dtypes):
		('float', [])        : 4 | ['FLOOR_AREA_SQM', 'coe_quota_premium_mean', 'mrt_dist_nearest_km', 'mall_dist_nearest_km']
		('int', [])          : 4 | ['approval_date_num', 'approval_month', 'YEAR_COMPLETED', 'MAX_F

Saved compact_no_coordinates submission to /Users/metul/Desktop/KDDM-project/exploration/metul/exp_7_autogluon_compact_no_coordinates_submission.csv


	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
		Fitting CategoryFeatureGenerator...
			Fitting CategoryMemoryMinimizeFeatureGenerator...
		Fitting TextSpecialFeatureGenerator...
			Fitting BinnedFeatureGenerator...
			Fitting DropDuplicatesFeatureGenerator...
		Fitting TextNgramFeatureGenerator...
			Fitting CountVectorizer for text features: ['STREET']
			CountVectorizer fit with vocabulary size = 9
		Reducing Vectorizer vocab size from 9 to 2 to avoid OOM error
	Stage 4 Generators:
		Fitting DropUniqueFeatureGenerator...
	Stage 5 Generators:
		Fitting DropDuplicatesFeatureGenerator...
	Types of features in original data (raw dtype, special dtypes):
		('float', [])        :  9 | ['FLOOR_AREA_SQM', 'coe_quota_premium_mean', 'LATITUDE', 'LONGITUDE', 'mrt_dist_nearest_km', ...]
		('int', [])          : 10 | ['approval_date_num', 'approval_month', 'YEAR_COMPLETED'

Saved compact_plus_candidates submission to /Users/metul/Desktop/KDDM-project/exploration/metul/exp_7_autogluon_compact_plus_candidates_submission.csv


,Id,full_preprocessed,compact_nonredundant,compact_no_coordinates,compact_plus_candidates
0,0,2790.023438,2773.601807,2765.013672,2759.389404
1,1,3642.981445,3644.737061,3618.741455,3669.272949
2,2,3185.030029,3211.164307,3230.654297,3226.633057
3,3,3134.428223,3149.320801,3131.618408,3162.478760
4,4,3268.248291,3253.361572,3240.814453,3281.963623
